### Title: N-Gram Language Models: Theory and Implementation

#### Objective:
In this notebook, we aim to:
- Understand the concept of N-gram language models in natural language processing (NLP).
- Explore their theoretical foundations, including mathematical representations.
- Discuss their advantages and drawbacks.
- Implement N-gram models (unigrams, bigrams, trigrams) from scratch and using the NLTK library in Python.
- Apply these models to practical tasks like text generation and probability calculation.
- Create a well-documented project suitable for GitHub, showcasing both educational and practical aspects.

#### Dataset:
We will use the "Emma" novel by Jane Austen, available in the Gutenberg corpus from the NLTK library. This dataset provides a rich collection of English text, ideal for building and testing N-gram language models.

---

### Section 1: Introduction to Language Models

A **language model** is a probabilistic framework that assigns probabilities to sequences of words, capturing how likely a sequence is in a given language. Language models are essential in NLP tasks such as:
- Machine translation
- Speech recognition
- Text generation
- Spell checking

Formally, a language model estimates the joint probability of a sequence of words \( w_1, w_2, \ldots, w_n \), denoted as \( P(w_1, w_2, \ldots, w_n) \). N-gram models are a simple yet effective type of language model that we'll explore in this notebook.

---

### Section 2: What are N-Grams?

**N-grams** are contiguous sequences of \( N \) items (typically words) extracted from a text. They represent local contextual patterns in language. Here are the main types we'll cover:
- **Unigram (N=1)**: A single word, e.g., "the", "cat".
- **Bigram (N=2)**: A pair of consecutive words, e.g., "the cat", "cat runs".
- **Trigram (N=3)**: A triplet of consecutive words, e.g., "the cat runs", "cat runs fast".

N-grams approximate the probability of a word based on a fixed number of preceding words, making them computationally manageable while capturing some linguistic structure.

---

### Section 3: Mathematical Representation

The probability of a word sequence \( w_1, w_2, \ldots, w_n \) can be decomposed using the **chain rule** of probability:

\[
P(w_1, w_2, \ldots, w_n) = P(w_1) \cdot P(w_2 | w_1) \cdot P(w_3 | w_1, w_2) \cdot \ldots \cdot P(w_n | w_1, \ldots, w_{n-1})
\]

Computing \( P(w_i | w_1, \ldots, w_{i-1}) \) directly is impractical due to the vast number of possible sequences. N-gram models simplify this by assuming the probability of a word depends only on the previous \( N-1 \) words (Markov assumption):

- **Unigram**: \( P(w_i) \)
- **Bigram**: \( P(w_i | w_{i-1}) \)
- **Trigram**: \( P(w_i | w_{i-2}, w_{i-1}) \)
- **General N-gram**: \( P(w_i | w_{i-N+1}, \ldots, w_{i-1}) \)

These probabilities are typically estimated using **Maximum Likelihood Estimation (MLE)** from a training corpus:
- Unigram: \( P(w_i) = \frac{\text{count}(w_i)}{\text{total words}} \)
- Bigram: \( P(w_i | w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})} \)
- Trigram: \( P(w_i | w_{i-2}, w_{i-1}) = \frac{\text{count}(w_{i-2}, w_{i-1}, w_i)}{\text{count}(w_{i-2}, w_{i-1})} \)

However, MLE can assign zero probability to unseen N-grams, so smoothing techniques are often applied (e.g., Laplace smoothing).

---

### Section 4: Advantages and Drawbacks

#### Advantages:
- **Simplicity**: N-gram models are straightforward to understand and implement.
- **Interpretability**: The probabilities are directly derived from frequency counts, making them easy to inspect.
- **Efficiency**: For small \( N \), they require minimal computational resources.
- **Versatility**: Applicable to tasks like text generation, spell checking, and more.

#### Drawbacks:
- **Data Sparsity**: As \( N \) increases, many N-grams may not appear in the training data, leading to zero probabilities.
- **Limited Context**: Only captures dependencies within \( N-1 \) words, missing longer-range relationships.
- **Storage Overhead**: Large \( N \) values result in an exponential number of N-grams, increasing memory use.
- **Overfitting**: Higher-order N-grams may overfit to the training corpus.

To mitigate sparsity, smoothing methods adjust probabilities for unseen N-grams, which we'll implement later.

---

### Section 5: Practical Implementation

#### a. Loading and Preprocessing the Data

We start by loading the "Emma" novel and preprocessing the text to ensure consistency.


In [1]:
import nltk
from nltk.corpus import gutenberg

# Download the Gutenberg corpus if not already downloaded
nltk.download('gutenberg')

# Load the Emma novel
corpus = gutenberg.words('austen-emma.txt')

# Preprocess: lowercase and keep only alphabetic words
words = [word.lower() for word in corpus if word.isalpha()]

print(f"Total words after preprocessing: {len(words)}")


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


Total words after preprocessing: 161600


##### Explanation:
We import NLTK and load the "Emma" text from the Gutenberg corpus. The preprocessing step converts all words to lowercase and filters out non-alphabetic tokens (e.g., punctuation, numbers), resulting in a clean list of words. We print the total number of words to verify the dataset size.



#### b. Building Unigram Model from Scratch

We compute unigram frequencies and probabilities.


In [2]:
from collections import Counter

# Calculate unigram counts
unigram_counts = Counter(words)

# Total number of words
total_words = len(words)

# Calculate unigram probabilities
unigram_probs = {word: count / total_words for word, count in unigram_counts.items()}

# Display top 5 unigrams
top_unigrams = sorted(unigram_probs.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 unigrams and their probabilities:")
for word, prob in top_unigrams:
    print(f"{word}: {prob:.5f}")


Top 5 unigrams and their probabilities:
to: 0.03242
the: 0.03218
and: 0.03030
of: 0.02655
i: 0.01967


##### Explanation:
Using `Counter`, we count the frequency of each word in the corpus. The unigram probability is the word's frequency divided by the total number of words. We sort the unigrams by probability and display the top 5, providing insight into the most common words in "Emma" (e.g., "the", "to").


#### c. Building Bigram Model from Scratch

We generate bigrams, compute their probabilities, and apply Laplace smoothing.



In [3]:
from collections import defaultdict

# Generate bigrams
bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]

# Calculate bigram counts
bigram_counts = Counter(bigrams)

# Use unigram counts for denominators
word_counts = unigram_counts

# Calculate bigram probabilities (MLE)
bigram_probs = defaultdict(dict)
for (w1, w2), count in bigram_counts.items():
    bigram_probs[w1][w2] = count / word_counts[w1]

# Laplace smoothing
vocab_size = len(set(words))
bigram_probs_smoothed = defaultdict(lambda: defaultdict(lambda: 1 / (word_counts.get(w1, 0) + vocab_size)))
for (w1, w2), count in bigram_counts.items():
    bigram_probs_smoothed[w1][w2] = (count + 1) / (word_counts[w1] + vocab_size)

# Example: Probability of "very" after "was"
w1, w2 = "was", "very"
prob_mle = bigram_probs.get(w1, {}).get(w2, 0)
prob_smoothed = bigram_probs_smoothed[w1][w2]
print(f"P({w2}|{w1}) (MLE): {prob_mle:.5f}")
print(f"P({w2}|{w1}) (Laplace): {prob_smoothed:.5f}")


P(very|was) (MLE): 0.03461
P(very|was) (Laplace): 0.00886


##### Explanation:
We create bigrams by pairing consecutive words and count their occurrences with `Counter`. The MLE probability \( P(w_2 | w_1) \) is the bigram count divided by the count of \( w_1 \). For Laplace smoothing, we add 1 to each bigram count and adjust the denominator by the vocabulary size, ensuring non-zero probabilities for unseen bigrams. We test this with "was very", comparing MLE (which might be zero if unseen) and smoothed probabilities.


#### d. Building Trigram Model from Scratch

We extend the approach to trigrams.


In [4]:
# Generate trigrams
trigrams = [(words[i], words[i+1], words[i+2]) for i in range(len(words)-2)]

# Calculate trigram counts
trigram_counts = Counter(trigrams)

# Calculate bigram counts for denominators
bigram_counts_for_trigrams = Counter([(words[i], words[i+1]) for i in range(len(words)-1)])

# Calculate trigram probabilities (MLE)
trigram_probs = defaultdict(lambda: defaultdict(dict))
for (w1, w2, w3), count in trigram_counts.items():
    trigram_probs[w1][w2][w3] = count / bigram_counts_for_trigrams[(w1, w2)]

# Example: Probability of "much" after "was very"
w1, w2, w3 = "was", "very", "much"
prob = trigram_probs.get(w1, {}).get(w2, {}).get(w3, 0)
print(f"P({w3}|{w1}, {w2}): {prob:.5f}")


P(much|was, very): 0.20482


##### Explanation:
Trigrams are generated from three consecutive words, and their counts are computed. The probability \( P(w_3 | w_1, w_2) \) uses the count of the bigram \( (w_1, w_2) \) as the denominator. We test with "was very much". Note that we omit smoothing here for brevity, but it follows the same principle as bigrams.


#### e. Applications

##### Text Generation

We generate text using the bigram model.


In [33]:
import random
import nltk
from collections import defaultdict

# Download NLTK resources (if not already installed)
nltk.download('punkt')

# Sample training text
corpus = """The cat sat on the mat. The dog barked at the cat. The mat was soft and warm.
            The cat purred happily. The dog wagged its tail. The cat and dog were friends."""

# Tokenize the text
tokens = nltk.word_tokenize(corpus)

# Create a bigram model (dictionary of word pairs)
bigrams = list(nltk.bigrams(tokens))
bigram_model = defaultdict(list)

for w1, w2 in bigrams:
    bigram_model[w1].append(w2)

# Function to generate text using the bigram model
def generate_text(start_word, length=10):
    if start_word not in bigram_model:
        return "Start word not found in corpus!"

    current_word = start_word
    sentence = [current_word]

    for _ in range(length - 1):
        next_words = bigram_model.get(current_word, None)
        if not next_words:
            break
        current_word = random.choice(next_words)
        sentence.append(current_word)

    return ' '.join(sentence)

# Generate text starting from "The"
print(generate_text("The", length=15))


The cat sat on the mat was soft and warm . The cat purred happily


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Section 6: Conclusion

In this notebook, we’ve comprehensively explored N-gram language models:
- **Theory**: Defined unigrams, bigrams, and trigrams, and derived their mathematical representations using the chain rule and MLE.
- **Pros and Cons**: Highlighted simplicity and sparsity issues, mitigated by smoothing.
- **Implementation**: Built models from scratch and with NLTK, applying them to text generation and probability calculation.

The "Emma" dataset provided a practical foundation, revealing common patterns in Austen’s writing. While N-gram models are foundational, they lack long-range context, a limitation addressed by modern neural models (e.g., transformers). Nonetheless, their simplicity makes them valuable for many tasks.
